In [4]:
import subprocess
subprocess.run(["pip", "install", "pandas", "numpy", "matplotlib"])

CompletedProcess(args=['pip', 'install', 'pandas', 'numpy', 'matplotlib'], returncode=0)

In [5]:
import pandas as pd
import numpy as np

# Load both datasets
matches = pd.read_csv("matches.csv")
deliveries = pd.read_csv("deliveries.csv")

# Check how many rows each has
print("Matches shape:", matches.shape)
print("Deliveries shape:", deliveries.shape)

Matches shape: (1095, 20)
Deliveries shape: (260920, 17)


In [6]:
# Merge deliveries with match info on match_id
df = pd.merge(deliveries, matches[['id','season','city','venue','winner','team1','team2']], 
              left_on='match_id', right_on='id', how='left')

print("Merged dataframe shape:", df.shape)
print(df.head())

Merged dataframe shape: (260920, 24)
   match_id  inning           batting_team                 bowling_team  over  \
0    335982       1  Kolkata Knight Riders  Royal Challengers Bangalore     0   
1    335982       1  Kolkata Knight Riders  Royal Challengers Bangalore     0   
2    335982       1  Kolkata Knight Riders  Royal Challengers Bangalore     0   
3    335982       1  Kolkata Knight Riders  Royal Challengers Bangalore     0   
4    335982       1  Kolkata Knight Riders  Royal Challengers Bangalore     0   

   ball       batter   bowler  non_striker  batsman_runs  ...  \
0     1   SC Ganguly  P Kumar  BB McCullum             0  ...   
1     2  BB McCullum  P Kumar   SC Ganguly             0  ...   
2     3  BB McCullum  P Kumar   SC Ganguly             0  ...   
3     4  BB McCullum  P Kumar   SC Ganguly             0  ...   
4     5  BB McCullum  P Kumar   SC Ganguly             0  ...   

   player_dismissed  dismissal_kind fielder      id   season       city  \
0         

In [7]:
# Check for missing values
print("Missing values:\n", df.isnull().sum())

# Drop rows where critical columns are missing
df = df.dropna(subset=['batter', 'bowler', 'batting_team'])

# Standardize team names (some teams changed names over the years)
df['batting_team'] = df['batting_team'].replace({
    'Delhi Daredevils': 'Delhi Capitals',
    'Deccan Chargers': 'Sunrisers Hyderabad',
    'Rising Pune Supergiants': 'Rising Pune Supergiant'
})

df['bowling_team'] = df['bowling_team'].replace({
    'Delhi Daredevils': 'Delhi Capitals',
    'Deccan Chargers': 'Sunrisers Hyderabad',
    'Rising Pune Supergiants': 'Rising Pune Supergiant'
})

print("After cleaning:", df.shape)

Missing values:
 match_id                 0
inning                   0
batting_team             0
bowling_team             0
over                     0
ball                     0
batter                   0
bowler                   0
non_striker              0
batsman_runs             0
extra_runs               0
total_runs               0
extras_type         246795
is_wicket                0
player_dismissed    247970
dismissal_kind      247970
fielder             251566
id                       0
season                   0
city                 12397
venue                    0
winner                 490
team1                    0
team2                    0
dtype: int64
After cleaning: (260920, 24)


In [8]:
# ------- BATSMAN STATS -------
batsman_stats = df.groupby('batter').agg(
    runs_scored = ('batsman_runs', 'sum'),
    balls_faced = ('ball', 'count')
).reset_index()

# Strike Rate = (runs / balls) * 100
batsman_stats['strike_rate'] = np.round(
    (batsman_stats['runs_scored'] / batsman_stats['balls_faced']) * 100, 2
)

# Filter batsmen with at least 100 balls faced
batsman_stats = batsman_stats[batsman_stats['balls_faced'] >= 100]
print("Top 10 batsmen by Strike Rate:")
print(batsman_stats.sort_values('strike_rate', ascending=False).head(10))

Top 10 batsmen by Strike Rate:
              batter  runs_scored  balls_faced  strike_rate
234  J Fraser-McGurk          330          150       220.00
652         WG Jacks          230          133       172.93
433          PD Salt          653          385       169.61
606         T Stubbs          405          239       169.46
617          TM Head          772          458       168.56
39        AD Russell         2488         1515       164.22
105      BCJ Cutting          238          146       163.01
208        H Klaasen          993          613       161.99
503  Ramandeep Singh          170          106       160.38
85   Ashutosh Sharma          189          118       160.17


In [22]:
# Recreate bowler_stats with wickets included
bowler_stats = df.groupby(['bowler', 'bowling_team', 'venue']).agg(
    runs_conceded = ('total_runs', 'sum'),
    balls_bowled  = ('ball', 'count'),
    wickets       = ('is_wicket', 'sum')   # this counts every wicket taken
).reset_index()

# Compute economy rate
bowler_stats['economy_rate'] = np.round(
    (bowler_stats['runs_conceded'] / bowler_stats['balls_bowled']) * 6, 2
)

print(bowler_stats.head())
print("Columns:", bowler_stats.columns.tolist())

           bowler         bowling_team  \
0  A Ashish Reddy  Sunrisers Hyderabad   
1  A Ashish Reddy  Sunrisers Hyderabad   
2  A Ashish Reddy  Sunrisers Hyderabad   
3  A Ashish Reddy  Sunrisers Hyderabad   
4  A Ashish Reddy  Sunrisers Hyderabad   

                                               venue  runs_conceded  \
0                                   Barabati Stadium             32   
1  Dr. Y.S. Rajasekhara Reddy ACA-VDCA Cricket St...             21   
2                                       Eden Gardens             15   
3                              M Chinnaswamy Stadium             86   
4                    MA Chidambaram Stadium, Chepauk             31   

   balls_bowled  wickets  economy_rate  
0            19        1         10.11  
1            19        1          6.63  
2            12        1          7.50  
3            48        2         10.75  
4            18        1         10.33  
Columns: ['bowler', 'bowling_team', 'venue', 'runs_conceded', 'balls_bowle

In [10]:
import sqlite3

# Create a local SQL database file
conn = sqlite3.connect('ipl_analysis.db')

# Save dataframes as SQL tables
df.to_sql('deliveries_full', conn, if_exists='replace', index=False)
batsman_stats.to_sql('batsman_stats', conn, if_exists='replace', index=False)
bowler_stats.to_sql('bowler_stats', conn, if_exists='replace', index=False)
matches.to_sql('matches', conn, if_exists='replace', index=False)

print("Tables loaded into SQL successfully")

Tables loaded into SQL successfully


In [12]:
# Query 1 — Top 10 batsmen by strike rate
query1 = pd.read_sql_query("""
    SELECT batter, runs_scored, balls_faced, strike_rate
    FROM batsman_stats
    ORDER BY strike_rate DESC
    LIMIT 10
""", conn)
print("Top 10 Batsmen by Strike Rate:\n", query1)

Top 10 Batsmen by Strike Rate:
             batter  runs_scored  balls_faced  strike_rate
0  J Fraser-McGurk          330          150       220.00
1         WG Jacks          230          133       172.93
2          PD Salt          653          385       169.61
3         T Stubbs          405          239       169.46
4          TM Head          772          458       168.56
5       AD Russell         2488         1515       164.22
6      BCJ Cutting          238          146       163.01
7        H Klaasen          993          613       161.99
8  Ramandeep Singh          170          106       160.38
9  Ashutosh Sharma          189          118       160.17


In [13]:
# Query 2 — Team head-to-head win rates
query2 = pd.read_sql_query("""
    SELECT team1, team2, COUNT(*) as total_matches,
           SUM(CASE WHEN winner = team1 THEN 1 ELSE 0 END) as team1_wins,
           SUM(CASE WHEN winner = team2 THEN 1 ELSE 0 END) as team2_wins
    FROM matches
    GROUP BY team1, team2
    ORDER BY total_matches DESC
    LIMIT 15
""", conn)
print("Head to Head:\n", query2)

Head to Head:
                           team1                        team2  total_matches  \
0           Chennai Super Kings               Mumbai Indians             22   
1   Royal Challengers Bangalore        Kolkata Knight Riders             19   
2   Royal Challengers Bangalore               Mumbai Indians             19   
3         Kolkata Knight Riders               Mumbai Indians             18   
4   Royal Challengers Bangalore          Chennai Super Kings             18   
5           Chennai Super Kings             Rajasthan Royals             17   
6                Mumbai Indians             Rajasthan Royals             17   
7         Kolkata Knight Riders             Rajasthan Royals             16   
8                Mumbai Indians        Kolkata Knight Riders             16   
9         Kolkata Knight Riders          Chennai Super Kings             15   
10        Kolkata Knight Riders              Kings XI Punjab             15   
11               Mumbai Indians      

In [16]:
# Check what columns bowler_stats actually has
print(bowler_stats.columns.tolist())

['bowler', 'venue', 'runs_conceded', 'balls_bowled', 'economy_rate']


In [17]:
# Query 3 — Venue-wise economy rate (fixed)
query3 = pd.read_sql_query("""
    SELECT venue, ROUND(AVG(economy_rate),2) as avg_economy
    FROM bowler_stats
    GROUP BY venue
    ORDER BY avg_economy ASC
    LIMIT 15
""", conn)
print("Venue-wise Economy:\n", query3)

Venue-wise Economy:
                                                 venue  avg_economy
0                                     OUTsurance Oval         6.96
1                          Subrata Roy Sahara Stadium         7.13
2                                       Nehru Stadium         7.37
3        Vidarbha Cricket Association Stadium, Jamtha         7.37
4                                        Buffalo Park         7.43
5                               De Beers Diamond Oval         7.43
6                                           Kingsmead         7.43
7                                    St George's Park         7.46
8                              MA Chidambaram Stadium         7.47
9                          Dr DY Patil Sports Academy         7.53
10                              New Wanderers Stadium         7.63
11                               Sheikh Zayed Stadium         7.83
12  Bharat Ratna Shri Atal Bihari Vajpayee Ekana C...         7.86
13                                       

In [18]:
# Recreate bowler_stats with bowling_team included
bowler_stats = df.groupby(['bowler', 'bowling_team', 'venue']).agg(
    runs_conceded = ('total_runs', 'sum'),
    balls_bowled  = ('ball', 'count')
).reset_index()

bowler_stats['economy_rate'] = np.round(
    (bowler_stats['runs_conceded'] / bowler_stats['balls_bowled']) * 6, 2
)

# Save updated table to SQL
bowler_stats.to_sql('bowler_stats', conn, if_exists='replace', index=False)
print("bowler_stats updated. Columns:", bowler_stats.columns.tolist())

bowler_stats updated. Columns: ['bowler', 'bowling_team', 'venue', 'runs_conceded', 'balls_bowled', 'economy_rate']


In [19]:
query3 = pd.read_sql_query("""
    SELECT venue, bowling_team, ROUND(AVG(economy_rate),2) as avg_economy
    FROM bowler_stats
    GROUP BY venue, bowling_team
    ORDER BY avg_economy ASC
    LIMIT 15
""", conn)
print("Venue-wise Economy:\n", query3)

Venue-wise Economy:
                                                 venue  \
0                              MA Chidambaram Stadium   
1                                       Nehru Stadium   
2   Bharat Ratna Shri Atal Bihari Vajpayee Ekana C...   
3                                        Buffalo Park   
4                          Subrata Roy Sahara Stadium   
5             Maharashtra Cricket Association Stadium   
6                    Zayed Cricket Stadium, Abu Dhabi   
7                      Sawai Mansingh Stadium, Jaipur   
8                          Subrata Roy Sahara Stadium   
9                                           Kingsmead   
10  Dr. Y.S. Rajasekhara Reddy ACA-VDCA Cricket St...   
11                                       Eden Gardens   
12                       Sardar Patel Stadium, Motera   
13                                   St George's Park   
14                             MA Chidambaram Stadium   

                   bowling_team  avg_economy  
0   Royal Challenge

In [20]:
# Query 4 — Season-wise run totals
query4 = pd.read_sql_query("""
    SELECT season, SUM(total_runs) as total_runs, COUNT(DISTINCT match_id) as matches_played
    FROM deliveries_full
    GROUP BY season
    ORDER BY season
""", conn)
print("Season-wise trends:\n", query4)

# Note this number — it confirms "16 seasons" for your resume
print("Total seasons:", query4.shape[0])

Season-wise trends:
      season  total_runs  matches_played
0   2007/08       17937              58
1      2009       16353              57
2   2009/10       18883              60
3      2011       21154              73
4      2012       22453              74
5      2013       22602              76
6      2014       18931              60
7      2015       18353              59
8      2016       18862              60
9      2017       18786              59
10     2018       19901              60
11     2019       19434              60
12  2020/21       19416              60
13     2021       18637              60
14     2022       24395              74
15     2023       25688              74
16     2024       25971              71
Total seasons: 17


In [21]:
# Export cleaned data for Power BI
query1.to_csv('top_batsmen.csv', index=False)
query2.to_csv('head_to_head.csv', index=False)
query3.to_csv('venue_economy.csv', index=False)
query4.to_csv('season_trends.csv', index=False)
batsman_stats.to_csv('batsman_full.csv', index=False)
bowler_stats.to_csv('bowler_full.csv', index=False)

print("All CSV files saved to IPL_Project folder")

All CSV files saved to IPL_Project folder


In [23]:
# Update SQL table
bowler_stats.to_sql('bowler_stats', conn, if_exists='replace', index=False)

# Re-export CSV for Power BI
bowler_stats.to_csv('bowler_full.csv', index=False)

print("Done — bowler_full.csv now includes wickets column")

Done — bowler_full.csv now includes wickets column
